# Context-bias audit of acoustic units

In [ ]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "configs" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. "
        "Expected to find configs/config.yaml in the project root."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f) or {}

OUTPUT_DIR = PROJECT_ROOT / CONFIG.get("paths", {}).get("output_dir", "output")
DICT_DIR = OUTPUT_DIR / CONFIG.get("dictionary", {}).get("output_subdir", "06_dictionary")
AUDIT_DIR = OUTPUT_DIR / CONFIG.get("audits", {}).get("output_subdir", "07_audits")
LOG_DIR = OUTPUT_DIR / CONFIG.get("paths", {}).get("log_dir", "output/logs").replace("output/", "")

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = LOG_DIR / "06_audit_unit_bias.log"

logger = logging.getLogger("06_audit_unit_bias")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(stream_handler)

logger.info("PROJECT_ROOT=%s", PROJECT_ROOT)
logger.info("CONFIG_PATH=%s", CONFIG_PATH)
logger.info("DICT_DIR=%s", DICT_DIR)
logger.info("AUDIT_DIR=%s", AUDIT_DIR)


In [ ]:
INVENTORY_PATH = DICT_DIR / "unit_inventory.csv"
CONTEXT_ASSOC_PATH = DICT_DIR / "unit_context_associations.csv"
CATEGORY_ASSOC_PATH = DICT_DIR / "unit_category_associations.csv"

assert INVENTORY_PATH.exists(), f"Missing: {INVENTORY_PATH}"
assert CONTEXT_ASSOC_PATH.exists(), f"Missing: {CONTEXT_ASSOC_PATH}"
assert CATEGORY_ASSOC_PATH.exists(), f"Missing: {CATEGORY_ASSOC_PATH}"

unit_inventory = pd.read_csv(INVENTORY_PATH)
context_assoc = pd.read_csv(CONTEXT_ASSOC_PATH)
category_assoc = pd.read_csv(CATEGORY_ASSOC_PATH)

logger.info("Loaded unit_inventory rows=%d", len(unit_inventory))
logger.info("Loaded context_assoc rows=%d", len(context_assoc))
logger.info("Loaded category_assoc rows=%d", len(category_assoc))

display(unit_inventory.head(3))
display(context_assoc.head(3))


In [ ]:
def get_config_value(path_list, default):
    cur = CONFIG
    for key in path_list:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

bias_cfg = get_config_value(["audits"], {})
if not bias_cfg:
    bias_cfg = get_config_value(["audit"], {})

dominant_warning_threshold = float(bias_cfg.get("dominant_warning_threshold", 0.80))
dominant_high_threshold = float(bias_cfg.get("dominant_high_threshold", 0.90))
min_segments_for_flag = int(bias_cfg.get("min_segments_for_flag", 8))

context_field_to_output = {
    "farm": AUDIT_DIR / "unit_farm_bias.csv",
    "barn_zone": AUDIT_DIR / "unit_zone_bias.csv",
    "microphone": AUDIT_DIR / "unit_microphone_bias.csv",
    "mic_placement_context": AUDIT_DIR / "unit_placement_bias.csv",
}

dominant_count_cols = {
    "farm": "dominant_farm_count",
    "barn_zone": "dominant_barn_zone_count",
    "microphone": "dominant_microphone_count",
    "mic_placement_context": "dominant_mic_placement_context_count",
}

dominant_value_cols = {
    "farm": "dominant_farm",
    "barn_zone": "dominant_barn_zone",
    "microphone": "dominant_microphone",
    "mic_placement_context": "dominant_mic_placement_context",
}

def safe_value(x):
    if pd.isna(x):
        return "NA"
    s = str(x).strip()
    return s if s else "NA"

def classify_bias_level(purity, segment_count):
    if pd.isna(purity):
        return "unknown"
    if segment_count < min_segments_for_flag:
        return "small_sample"
    if purity >= dominant_high_threshold:
        return "high"
    if purity >= dominant_warning_threshold:
        return "moderate"
    return "low"


In [ ]:
def build_bias_table(field_name: str) -> pd.DataFrame:
    field_df = context_assoc.loc[context_assoc["context_field"] == field_name].copy()
    field_df["context_value"] = field_df["context_value"].map(safe_value)

    unit_base = unit_inventory[[
        "unit_id",
        "unit_index",
        "segment_count",
        "clip_count",
        "dominant_main_category_final",
        "dominant_sub_category_final",
        dominant_value_cols[field_name],
        dominant_count_cols[field_name],
    ]].copy()

    summary = (
        field_df
        .sort_values(["unit_id", "count", "lift_vs_global"], ascending=[True, False, False])
        .groupby("unit_id", as_index=False)
        .first()
        .rename(columns={
            "context_value": "top_context_value",
            "count": "top_context_count",
            "unit_proportion": "top_context_proportion",
            "lift_vs_global": "top_context_lift_vs_global",
            "global_proportion": "top_context_global_proportion",
            "rank_within_unit_by_count": "top_rank_by_count",
            "rank_within_unit_by_lift": "top_rank_by_lift",
        })
    )

    out = unit_base.merge(summary[[
        "unit_id",
        "top_context_value",
        "top_context_count",
        "top_context_proportion",
        "top_context_lift_vs_global",
        "top_context_global_proportion",
        "top_rank_by_count",
        "top_rank_by_lift",
    ]], on="unit_id", how="left")

    out["dominant_context_value"] = out[dominant_value_cols[field_name]].map(safe_value)
    out["dominant_context_count"] = out[dominant_count_cols[field_name]]
    out["dominant_context_proportion"] = out["dominant_context_count"] / out["segment_count"].replace(0, np.nan)

    out["bias_level"] = [
        classify_bias_level(purity, segs)
        for purity, segs in zip(out["dominant_context_proportion"], out["segment_count"])
    ]

    out["flag_context_locked"] = (
        (out["segment_count"] >= min_segments_for_flag) &
        (out["dominant_context_proportion"] >= dominant_warning_threshold)
    )

    out["flag_highly_locked"] = (
        (out["segment_count"] >= min_segments_for_flag) &
        (out["dominant_context_proportion"] >= dominant_high_threshold)
    )

    # Effective spread across context values using normalized entropy
    entropy_df = field_df.copy()
    entropy_df["p"] = entropy_df["count"] / entropy_df["unit_total"].replace(0, np.nan)
    entropy = (
        entropy_df.groupby("unit_id")["p"]
        .apply(lambda s: float(-(s * np.log(s + 1e-12)).sum()))
        .reset_index(name="context_entropy")
    )

    nvals = field_df.groupby("unit_id")["context_value"].nunique().reset_index(name="n_context_values")
    out = out.merge(entropy, on="unit_id", how="left").merge(nvals, on="unit_id", how="left")

    out["normalized_context_entropy"] = out["context_entropy"] / np.log(out["n_context_values"].clip(lower=2))
    out["normalized_context_entropy"] = out["normalized_context_entropy"].clip(lower=0, upper=1)

    out["field_name"] = field_name

    ordered_cols = [
        "field_name",
        "unit_id",
        "unit_index",
        "segment_count",
        "clip_count",
        "dominant_main_category_final",
        "dominant_sub_category_final",
        "dominant_context_value",
        "dominant_context_count",
        "dominant_context_proportion",
        "top_context_value",
        "top_context_count",
        "top_context_proportion",
        "top_context_lift_vs_global",
        "top_context_global_proportion",
        "n_context_values",
        "context_entropy",
        "normalized_context_entropy",
        "bias_level",
        "flag_context_locked",
        "flag_highly_locked",
    ]
    return out[ordered_cols].sort_values(
        ["flag_highly_locked", "flag_context_locked", "dominant_context_proportion", "segment_count"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

audit_tables = {}
for field_name, output_path in context_field_to_output.items():
    bias_table = build_bias_table(field_name)
    audit_tables[field_name] = bias_table
    bias_table.to_csv(output_path, index=False)
    logger.info("Saved %s rows=%d -> %s", field_name, len(bias_table), output_path)

display(audit_tables["farm"].head(10))
display(audit_tables["barn_zone"].head(10))


In [ ]:
summary_rows = []
for field_name, df in audit_tables.items():
    summary_rows.append({
        "context_field": field_name,
        "n_units": int(len(df)),
        "n_moderate_or_higher": int(df["flag_context_locked"].sum()),
        "n_high": int(df["flag_highly_locked"].sum()),
        "median_dominant_context_proportion": float(df["dominant_context_proportion"].median()),
        "max_dominant_context_proportion": float(df["dominant_context_proportion"].max()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("n_high", ascending=False).reset_index(drop=True)
display(summary_df)
logger.info("Audit summary:\n%s", summary_df.to_string(index=False))


In [ ]:
REPORT_PATH = AUDIT_DIR / "unit_bias_report.md"

def bullet_units(df: pd.DataFrame, top_n: int = 8):
    rows = []
    for _, row in df.head(top_n).iterrows():
        rows.append(
            f"- {row['unit_id']}: dominant={row['dominant_context_value']} | "
            f"purity={row['dominant_context_proportion']:.3f} | "
            f"segments={int(row['segment_count'])} | clips={int(row['clip_count'])} | "
            f"main={safe_value(row['dominant_main_category_final'])} | "
            f"sub={safe_value(row['dominant_sub_category_final'])}"
        )
    return "\n".join(rows) if rows else "- None"

report_lines = []
report_lines.append("# Unit Bias Report")
report_lines.append("")
report_lines.append("This report is descriptive. It flags units that are strongly concentrated in one recording context.")
report_lines.append("It does not claim that such units are biologically meaningful or invalid by itself.")
report_lines.append("")
report_lines.append("## Audit thresholds")
report_lines.append(f"- Moderate/flag threshold: dominant context proportion >= {dominant_warning_threshold:.2f}")
report_lines.append(f"- High threshold: dominant context proportion >= {dominant_high_threshold:.2f}")
report_lines.append(f"- Minimum segments for flagging: {min_segments_for_flag}")
report_lines.append("")

report_lines.append("## Overall summary")
for _, row in summary_df.iterrows():
    report_lines.append(
        f"- {row['context_field']}: "
        f"{int(row['n_moderate_or_higher'])} flagged, "
        f"{int(row['n_high'])} high, "
        f"median dominant proportion={row['median_dominant_context_proportion']:.3f}, "
        f"max={row['max_dominant_context_proportion']:.3f}"
    )
report_lines.append("")

for field_name, df in audit_tables.items():
    report_lines.append(f"## {field_name}")
    report_lines.append("")
    report_lines.append("Top highly locked or flagged units:")
    locked = df.loc[df["flag_context_locked"]].copy()
    if locked.empty:
        report_lines.append("- No units crossed the warning threshold.")
    else:
        report_lines.append(bullet_units(locked.sort_values(
            ["flag_highly_locked", "dominant_context_proportion", "segment_count"],
            ascending=[False, False, False]
        )))
    report_lines.append("")
    report_lines.append("Interpretation note:")
    report_lines.append(
        f"- Units concentrated in {field_name} may reflect recording setup, environment, management routine, "
        f"or a real acoustic-context relationship. Treat them as audit flags, not biological conclusions."
    )
    report_lines.append("")

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

logger.info("Saved markdown report -> %s", REPORT_PATH)
print(REPORT_PATH)


In [ ]:
for field_name, df in audit_tables.items():
    print("=" * 90)
    print(field_name)
    print(df[[
        "unit_id",
        "segment_count",
        "clip_count",
        "dominant_context_value",
        "dominant_context_proportion",
        "bias_level",
        "flag_context_locked",
        "flag_highly_locked",
    ]].head(12).to_string(index=False))

print("\nSaved files:")
for path in [
    AUDIT_DIR / "unit_farm_bias.csv",
    AUDIT_DIR / "unit_zone_bias.csv",
    AUDIT_DIR / "unit_microphone_bias.csv",
    AUDIT_DIR / "unit_placement_bias.csv",
    AUDIT_DIR / "unit_bias_report.md",
    LOG_PATH,
]:
    print("-", path)
